In [ ]:
import sys
sys.path.append('./utils')
from utils import load_data, scale_data, load_and_prepare_kfold_data, evaluate_model, plot_residuals, save_results

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.model_selection import cross_val_score,train_test_split,GridSearchCV,KFold
from sklearn.pipeline import Pipeline
import seaborn as sns

In [ ]:
X_train_cv, X_val_cv, X_test_cv, y_train_cv, y_val_cv, y_test_cv = load_data()
X_train_cv, X_val_cv, X_test_cv, scaler = scale_data(X_train_cv, X_val_cv, X_test_cv)

In [ ]:
C_values       = [0.1, 1, 10, 100, 1000]
epsilon_values = [0.01, 0.1, 0.5, 1,3,]

best_val_rmse = float('inf')
best_C        = None
best_epsilon  = None
results       = []

for c in C_values:
    for e in epsilon_values:
        svr_cv = SVR(kernel='rbf', C=c, epsilon=e)
        svr_cv.fit(X_train_cv, y_train_cv)      
        RMSE = np.sqrt(mean_squared_error(y_train_cv, svr_cv.predict(X_train_cv)))
        results.append({'C': c, 'epsilon': e, 'val_rmse': RMSE})
        
        if RMSE < best_val_rmse:
            best_val_rmse = RMSE 
            best_C = c
            best_epsilon = e
            
print(f"Best C: {best_C}, Best epsilon: {best_epsilon}, Val RMSE: {best_val_rmse:.4f}")

results_df = pd.DataFrame(results)
pivot = results_df.pivot(index='C', columns='epsilon', values='val_rmse')

plt.figure(figsize=(8, 5))

sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd_r')
plt.title('SVR: Validation RMSE by C and Epsilon')
plt.show()

svr_cv = SVR(kernel='rbf', C=best_C, epsilon=best_epsilon)
svr_cv.fit(X_train_cv, y_train_cv)

y_pred_train_cv = svr_cv.predict(X_train_cv)
y_pred_val_cv   = svr_cv.predict(X_val_cv)

print("Train RMSE:", np.sqrt(mean_squared_error(y_train_cv, y_pred_train_cv)))
print("Train R²  :", r2_score(y_train_cv, y_pred_train_cv))
print("Val RMSE  :", np.sqrt(mean_squared_error(y_val_cv, y_pred_val_cv)))
print("Val R²    :", r2_score(y_val_cv, y_pred_val_cv))

## K-Folds Validation


In [ ]:
X_train_kf, X_test_kf, y_train_kf, y_test_kf = load_and_prepare_kfold_data(r"../Datasets/uci_concrete_data.csv")

In [ ]:
param_grid = {
    'svr__C':       [0.1, 1, 10, 100, 1000],
    'svr__epsilon': [0.01, 0.1, 0.5, 1,3,5],
    'svr__kernel':  ['rbf', 'linear']
}

pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('svr', SVR())
])

grid_search = GridSearchCV(pipeline, param_grid, 
                           cv=KFold(n_splits=3, shuffle=True, random_state=42),
                           scoring='neg_root_mean_squared_error',
                           n_jobs=-1)
grid_search.fit(X_train_kf, y_train_kf)

print("Best params:", grid_search.best_params_)
print("Best RMSE:",  -grid_search.best_score_)

# Training performance
y_pred_train_kf = grid_search.predict(X_train_kf)
print("\nTrain RMSE:", np.sqrt(mean_squared_error(y_train_kf, y_pred_train_kf)))
print("Train R²  :", r2_score(y_train_kf, y_pred_train_kf))

In [ ]:
# Testing performance
y_pred_kf = grid_search.predict(X_test_kf)
results = evaluate_model('SVR', y_train_kf, y_pred_train_kf, y_test_kf, y_pred_kf)
plot_residuals('SVR', y_test_kf, y_pred_kf)
save_results(results, 'SVR_results.csv')